# ProcessDataFrame: Frictionless Process Behavior Analysis

This notebook demonstrates the new `ProcessDataFrame` API that makes process behavior analysis intuitive and discoverable.

## Key Features

1. **Auto-completion** for column names via `data.columns.ColumnName`
2. **SDS-driven execution** - the data dictates what analyses are supported
3. **Automatic IMR charts** for simple series (SDS 0)
4. **Clear explanations** of what's running and why


In [ ]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame

# Set random seed for reproducibility
np.random.seed(42)

## Example 1: Simple Series (SDS 0) → IMR Chart

When you have a simple time series with no grouping structure, ProcessDataFrame automatically runs an **Individual Moving Range (IMR)** chart - just like qcc!

In [ ]:
# Create simple measurement series
simple_data = pd.DataFrame({
    'Measurement': np.random.normal(100, 2, 30),
    'Time': pd.date_range('2024-01-01', periods=30, freq='D')
})

# Wrap in ProcessDataFrame
data = ProcessDataFrame(simple_data)

# Auto-completion magic! Type `data.columns.` and see your columns appear
analysis = data.analyze(
    response_var=data.columns.Measurement,
    time_var=data.columns.Time
)

# Prints explanation of SDS detection and chosen analysis

In [ ]:
# Access results
result = analysis.calculate()
print("\nAnalysis Result:")
print(result)

## Example 2: Manufacturing Data with Operators (Grouped) → Xbar/S Charts

When you have rational subgroups, ProcessDataFrame detects the structure and automatically runs **Xbar and S charts** to track both location (mean) and variation.

In [ ]:
# Create manufacturing data with operators and machines
n_obs = 120
manufacturing_data = pd.DataFrame({
    'Height': np.random.normal(50, 3, n_obs),
    'Width': np.random.normal(25, 1, n_obs),
    'Operator': np.random.choice(['Alice', 'Bob', 'Charlie'], n_obs),
    'Machine': np.random.choice(['M1', 'M2', 'M3'], n_obs),
    'ProductionTime': pd.date_range('2024-01-01', periods=n_obs, freq='H')
})

data = ProcessDataFrame(manufacturing_data)

# Auto-completion works here too!
analysis = data.analyze(
    response_var=data.columns.Height,
    time_var=data.columns.ProductionTime,
    grouping_vars=[data.columns.Operator, data.columns.Machine]
)

# Notice: System explains it detected SDS 1 or 2 and why it's running Xbar/S

In [ ]:
# Get both Xbar and S chart results
result = analysis.calculate()

print("\nXbar Chart (Subgroup Means):")
print(result['Xbar']['data'].head())

print("\nS Chart (Subgroup Variation):")
print(result['Sbar']['data'].head())

## Example 3: Quality Data with Multiple Factors

ProcessDataFrame handles complex multi-factor designs and explains the detected Sampling Design State.

In [ ]:
# Create data with 2 factors and time
factors = pd.DataFrame({
    'Factor1': ['Low', 'High'] * 60,
    'Factor2': ['A', 'B', 'C', 'D'] * 30
})

quality_data = pd.DataFrame({
    'Strength': np.random.normal(100, 5, 120),
    'Temperature': np.random.choice(['Low', 'High'], 120),
    'Pressure': np.random.choice(['A', 'B'], 120),
    'Batch': range(1, 121)
})

data = ProcessDataFrame(quality_data)

analysis = data.analyze(
    response_var=data.columns.Strength,
    time_var=data.columns.Batch,
    grouping_vars=[data.columns.Temperature, data.columns.Pressure]
)

# System detects SDS and explains capabilities

## Example 4: Zero-Centered Analysis

Sometimes you want to center your data at zero to focus on deviations. ProcessDataFrame makes this easy.

In [ ]:
# Data that's not centered at zero
offset_data = pd.DataFrame({
    'Value': np.random.normal(1000, 10, 50),
    'Sequence': range(1, 51)
})

data = ProcessDataFrame(offset_data)

analysis = data.analyze(
    response_var=data.columns.Value,
    time_var=data.columns.Sequence,
    zero_center=True  # Subtract mean to focus on variation
)

result = analysis.calculate()
print("\nZero-centered analysis:")
print(result)

## Why This API is Better

### Before (Old API)
```python
# Easy to make typos in column names!
spec = {
    'analysis_type': 'Imr',  # User must know which type
    'response_var': 'Measurment',  # Typo! Will fail
    'time_var': 'Time',
    'rsg_vars': None
}
analysis = Analysis(df, spec)
```

### After (New API)
```python
# Auto-completion prevents typos!
# System chooses correct analysis type!
data = ProcessDataFrame(df)
analysis = data.analyze(
    response_var=data.columns.Measurement,  # IDE autocompletes
    time_var=data.columns.Time
)
# Prints: "Detected SDS 0: Running IMR Chart (Individual Moving Range)"
```

## Benefits

1. **No more typos** - IDE autocomplete ensures valid column names
2. **No more wrong analysis types** - System detects SDS and chooses appropriately
3. **Transparency** - Always explains what it's doing and why
4. **Follows the data** - Analysis adapts to your data structure
5. **Pythonic** - Clean, readable, discoverable API


## Understanding Sampling Design States (SDS)

ProcessDataFrame automatically detects which of 7 SDS categories your data falls into:

- **SDS 0**: No grouping or time → IMR chart
- **SDS 1**: Full replication (all cells n≥2) → Full VAS + Xbar/S
- **SDS 2**: No replication (all cells n=1) → Residuals + Xbar/S
- **SDS 3**: Partial replication (mixed) → Hybrid approach
- **SDS 4**: Single condition over time → Time series analysis
- **SDS 5**: Nested design → Variance components
- **SDS 6**: Irregular grid → Adaptive limits

You don't need to know these - the system figures it out and tells you!